# 11. Video QA 벤치마크 — AutoGaze ON vs OFF

> **NVILA + AutoGaze**를 8개 표준 비디오 QA 벤치마크에서 평가합니다.  
> 전체 실행은 `scripts/run_benchmarks.sh`를 사용하고, 이 노트북은 **인터랙티브 탐색**과 **스모크 테스트**를 위한 것입니다.

## 목차
1. 환경 설정
2. 지원 벤치마크 목록
3. NVILA + AutoGaze 로드
4. 단일 샘플 추론 데모
5. AutoGaze Gaze Map 시각화
6. 스모크 테스트 (N=20)
7. AutoGaze ON vs OFF 정확도 비교
8. Gazing Ratio 스윕
9. 지연 시간 분석
10. 전체 실행 가이드

---
## 1. 환경 설정

In [ ]:
import sys, os, warnings, time, json
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

# ── 한국어 폰트 ────────────────────────────────────────────────────────────
def _configure_mpl_cjk():
    matplotlib.rcParams["axes.unicode_minus"] = False
    try:
        import matplotlib.font_manager as fm
        for name in ["Apple SD Gothic Neo", "AppleGothic", "NanumGothic",
                     "Noto Sans CJK KR", "Noto Sans CJK JP", "DejaVu Sans"]:
            try:
                path = fm.findfont(fm.FontProperties(family=name), fallback_to_default=False)
                if path and os.path.exists(path):
                    matplotlib.rcParams["font.family"] = name
                    return name
            except Exception:
                pass
    except Exception:
        pass
    return "default"

font_name = _configure_mpl_cjk()

# ── 디바이스 ───────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def _sync():
    if device.type == "cuda":  torch.cuda.synchronize()
    elif device.type == "mps": torch.mps.synchronize()

print(f"device : {device}")
print(f"font   : {font_name}")

---
## 2. 지원 벤치마크 목록

In [ ]:
from autogaze.eval.tasks import TASKS

# 벤치마크 정보 표 출력
rows = []
for name, cfg in TASKS.items():
    has_bytes = "✓ 자동" if cfg.video_bytes_col else "✗ 로컬 필요"
    rows.append((name, cfg.hf_repo, cfg.hf_split, has_bytes))

print(f"{'Task 이름':22} {'HuggingFace 레포':38} {'Split':8} {'비디오 소스'}")
print("-" * 85)
for r in rows:
    print(f"{r[0]:22} {r[1]:38} {r[2]:8} {r[3]}")

---
## 3. NVILA + AutoGaze 로드

- **AutoGaze ON**: `gazing_ratio_tile=0.75` → 75% 패치 선택
- **AutoGaze OFF**: `autogaze_model_id` 미설정 → 100% 패치 (기준선)

> NVILA-8B는 약 **16~18 GB VRAM** 필요 (bfloat16 기준). `device_map="auto"`로 자동 분산.

In [ ]:
from transformers import AutoProcessor, AutoModel

MODEL_PATH = "../weights/NVILA-8B-HD-Video"
AG_PATH    = "../weights/AutoGaze"
RATIO      = 0.75

assert Path(MODEL_PATH).exists(), f"NVILA 가중치 없음: {MODEL_PATH}"
assert Path(AG_PATH).exists(),    f"AutoGaze 가중치 없음: {AG_PATH}"

print("[1/3] AutoGaze ON 프로세서 로드 중...")
t0 = time.perf_counter()
processor_ag = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    autogaze_model_id=AG_PATH,
    gazing_ratio_tile=RATIO,
    gazing_ratio_thumbnail=RATIO,
)
print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

print("[2/3] AutoGaze OFF 프로세서 로드 중...")
t0 = time.perf_counter()
processor_base = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    # autogaze_model_id 없음 = AutoGaze 비활성화
)
print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

print("[3/3] NVILA 모델 로드 중 (~16 GB)...")
t0 = time.perf_counter()
model = AutoModel.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

# AutoGaze 모델 직접 참조 (gaze map 시각화용)
ag_model = processor_ag._autogaze_model
print(f"\nAutoGaze 파라미터: {sum(p.numel() for p in ag_model.parameters()):,}")

---
## 4. 단일 샘플 추론 데모

VideoMME에서 샘플 1개를 가져와 AutoGaze ON/OFF 각각 추론합니다.

In [ ]:
from datasets import load_dataset
from autogaze.eval.run_benchmark import load_video_from_bytes
from autogaze.eval.tasks import TASKS

task_cfg = TASKS["videomme"]
print("VideoMME 데이터셋 로드 중 (첫 실행 시 다운로드)...")
ds_vmme = load_dataset(task_cfg.hf_repo, split=task_cfg.hf_split)
print(f"  총 샘플: {len(ds_vmme)}")

# 첫 번째 샘플 가져오기
sample = ds_vmme[0]
print(f"\n샘플 ID  : {sample.get('question_id', 0)}")
print(f"비디오 ID: {sample['videoID']}")
print(f"duration : {sample.get('duration', '?')}")
print(f"질문     : {sample['question']}")
print(f"선택지   : {sample['options']}")
print(f"정답     : {sample['answer']}")

# 비디오 프레임 로드 (HF bytes에서 직접)
NUM_FRAMES = 16
frames = load_video_from_bytes(sample[task_cfg.video_bytes_col], num_frames=NUM_FRAMES)
print(f"\n로드된 프레임: {len(frames)}장  ({frames[0].size[0]}×{frames[0].size[1]} px)")

In [ ]:
def run_nvila(processor, model, frames, prompt, max_new_tokens=16):
    inputs = processor(text=[prompt], videos=[frames], return_tensors="pt", padding=True)
    dev = next(model.parameters()).device
    inputs = {k: v.to(dev) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    with torch.inference_mode():
        gen_ids = model.generate(
            **{k: v for k, v in inputs.items() if k != "labels"},
            max_new_tokens=max_new_tokens,
            do_sample=False, temperature=None, top_p=None,
        )
    new_tok = gen_ids[0][inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(new_tok, skip_special_tokens=True)

prompt = task_cfg.build_prompt(sample)
print("=== 프롬프트 ===")
print(prompt)

# AutoGaze ON
_sync(); t0 = time.perf_counter()
ans_ag = run_nvila(processor_ag, model, frames, prompt)
_sync(); t_ag = time.perf_counter() - t0

# AutoGaze OFF
_sync(); t0 = time.perf_counter()
ans_base = run_nvila(processor_base, model, frames, prompt)
_sync(); t_base = time.perf_counter() - t0

gt = task_cfg.get_ground_truth(sample)
pred_ag   = task_cfg.parse_prediction(ans_ag)
pred_base = task_cfg.parse_prediction(ans_base)

print(f"\n정답          : {gt}")
print(f"AutoGaze ON   : {pred_ag:>3}  ({t_ag:.2f}s)  {'✓' if pred_ag==gt else '✗'}")
print(f"AutoGaze OFF  : {pred_base:>3}  ({t_base:.2f}s)  {'✓' if pred_base==gt else '✗'}")

---
## 5. AutoGaze Gaze Map 시각화

AutoGaze가 예측한 14×14 gaze map을 비디오 프레임에 overlay합니다.

In [ ]:
from autogaze.models.autogaze.processing_autogaze import AutoGazeImageProcessor
import torch.nn.functional as F

ag_proc = AutoGazeImageProcessor.from_pretrained(AG_PATH)
ag_model_eval = ag_model.eval()

def get_gaze_map(frames, ratio=0.75):
    """frames: list[PIL], returns gaze_map (T, 14, 14) numpy float32"""
    ag_dev = next(ag_model_eval.parameters()).device
    batch = ag_proc(images=frames, return_tensors="pt")["pixel_values"]  # (T, 3, 224, 224)
    batch = batch.unsqueeze(0).to(ag_dev)   # (1, T, 3, 224, 224)
    with torch.no_grad():
        out = ag_model_eval({"video": batch}, gazing_ratio=ratio)
    # gazing_mask[-1]: (1, T, 196) — finest 14×14 scale
    mask = out["gazing_mask"][-1][0].float()  # (T, 196)
    return mask.reshape(-1, 14, 14).cpu().numpy()

print("Gaze map 계산 중...")
gaze_maps = get_gaze_map(frames[:8], ratio=RATIO)  # 처음 8프레임
print(f"gaze_maps shape: {gaze_maps.shape}  (선택 비율 {gaze_maps.mean():.2f})")

In [ ]:
# ── Gaze Overlay 시각화 ────────────────────────────────────────────────────
N_SHOW = min(8, len(frames))
fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW * 2.2, 5))

for i in range(N_SHOW):
    # 원본 프레임
    axes[0, i].imshow(frames[i])
    axes[0, i].axis("off")
    axes[0, i].set_title(f"Frame {i}", fontsize=8)

    # Gaze overlay
    gm = gaze_maps[i]  # (14, 14)
    frame_np = np.array(frames[i].resize((224, 224)))

    # 선택 영역은 원본, 비선택 영역은 어둡게
    mask_up = F.interpolate(
        torch.tensor(gm).unsqueeze(0).unsqueeze(0).float(),
        size=(224, 224), mode="nearest"
    ).squeeze().numpy()  # (224, 224)

    overlay = frame_np.copy().astype(float)
    overlay[mask_up == 0] *= 0.25   # 비선택 = 어둡게
    overlay = overlay.clip(0, 255).astype(np.uint8)

    # 선택 경계선 (파란색)
    axes[1, i].imshow(overlay)
    axes[1, i].imshow(gm, cmap="cool", alpha=0.35,
                      extent=[0, 224, 224, 0], aspect="auto")
    n_sel = int(gm.sum())
    axes[1, i].set_title(f"Gaze ({n_sel}/196)", fontsize=8)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("원본", fontsize=9)
axes[1, 0].set_ylabel(f"Gaze (r={RATIO})", fontsize=9)

fig.suptitle(
    f"VideoMME: {sample['question'][:60]}…",
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.show()

---
## 6. 스모크 테스트 (N=20)

`autogaze.eval.run_benchmark.evaluate()`를 직접 호출합니다.  
빠른 검증용 — 전체 실행은 섹션 10의 스크립트를 사용하세요.

In [ ]:
from autogaze.eval.run_benchmark import evaluate
from pathlib import Path

N_SMOKE   = 20
OUT_DIR   = Path("../results/smoke")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SMOKE_TASKS = ["videomme", "mvbench", "nextqa"]   # ← 원하는 태스크 추가

smoke_results = {}  # task → {ag_acc, base_acc}

for task_name in SMOKE_TASKS:
    print(f"\n{'='*50}")
    print(f"Task: {task_name}  (N={N_SMOKE})")
    print(f"{'='*50}")

    # AutoGaze ON
    out_ag = OUT_DIR / f"{task_name}_ag_smoke.json"
    metrics_ag = evaluate(
        task_name     = task_name,
        video_dir     = None,
        model_path    = MODEL_PATH,
        autogaze_path = AG_PATH,
        gazing_ratio  = RATIO,
        num_frames    = NUM_FRAMES,
        max_new_tokens= 16,
        output_path   = out_ag,
        max_samples   = N_SMOKE,
        resume        = True,
    )

    # AutoGaze OFF
    out_base = OUT_DIR / f"{task_name}_baseline_smoke.json"
    metrics_base = evaluate(
        task_name     = task_name,
        video_dir     = None,
        model_path    = MODEL_PATH,
        autogaze_path = None,     # ← OFF
        gazing_ratio  = RATIO,
        num_frames    = NUM_FRAMES,
        max_new_tokens= 16,
        output_path   = out_base,
        max_samples   = N_SMOKE,
        resume        = True,
    )

    smoke_results[task_name] = {
        "ag"      : metrics_ag["overall_accuracy"],
        "baseline": metrics_base["overall_accuracy"],
    }

print("\n스모크 테스트 완료")

---
## 7. AutoGaze ON vs OFF 정확도 비교

In [ ]:
tasks_done   = list(smoke_results.keys())
acc_ag       = [smoke_results[t]["ag"]       for t in tasks_done]
acc_baseline = [smoke_results[t]["baseline"] for t in tasks_done]
diffs        = [a - b for a, b in zip(acc_ag, acc_baseline)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(tasks_done))
w = 0.35

# 왼쪽: 절대 정확도
axes[0].bar(x - w/2, acc_baseline, w, label="AutoGaze OFF", color="#607D8B", alpha=0.85)
axes[0].bar(x + w/2, acc_ag,       w, label=f"AutoGaze ON (r={RATIO})",
            color="#2196F3", alpha=0.85)
for i, (b, a) in enumerate(zip(acc_baseline, acc_ag)):
    axes[0].text(i-w/2, b+0.3, f"{b:.1f}", ha="center", va="bottom", fontsize=9, color="#455A64")
    axes[0].text(i+w/2, a+0.3, f"{a:.1f}", ha="center", va="bottom", fontsize=9, color="#1565C0")
axes[0].set_xticks(x)
axes[0].set_xticklabels(tasks_done, fontsize=10)
axes[0].set_ylabel("정확도 (%)")
axes[0].set_title(f"AutoGaze ON vs OFF  (N={N_SMOKE}/task)", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)

# 오른쪽: 차이 (AutoGaze - Baseline)
colors = ["#4CAF50" if d >= 0 else "#F44336" for d in diffs]
bars = axes[1].bar(x, diffs, color=colors, alpha=0.85)
for bar, d in zip(bars, diffs):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 d + (0.2 if d >= 0 else -0.5),
                 f"{d:+.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(tasks_done, fontsize=10)
axes[1].set_ylabel("정확도 차이 (AG - Baseline)")
axes[1].set_title("AutoGaze 효과 (+ = 향상)", fontsize=12)
axes[1].grid(axis="y", alpha=0.3)

note = f"※ N={N_SMOKE} 스모크 테스트 결과. 전체 실행: bash scripts/run_benchmarks.sh"
fig.text(0.5, -0.03, note, ha="center", fontsize=9, color="#555", style="italic")
plt.tight_layout()
plt.show()

---
## 8. Gazing Ratio 스윕

VideoMME에서 ratio를 바꾸며 정확도 변화를 측정합니다. (N=50)

In [ ]:
RATIOS_SWEEP = [0.25, 0.5, 0.75, 1.0]
N_SWEEP      = 50
SWEEP_TASK   = "videomme"

ratio_accs = {}

for r in RATIOS_SWEEP:
    label = "baseline" if r == 1.0 else f"ag{int(r*100):03d}"
    out_p = OUT_DIR / f"{SWEEP_TASK}_{label}_sweep.json"

    print(f"ratio={r} ...", end=" ", flush=True)
    metrics = evaluate(
        task_name     = SWEEP_TASK,
        video_dir     = None,
        model_path    = MODEL_PATH,
        autogaze_path = None if r == 1.0 else AG_PATH,
        gazing_ratio  = r,
        num_frames    = NUM_FRAMES,
        max_new_tokens= 16,
        output_path   = out_p,
        max_samples   = N_SWEEP,
        resume        = True,
    )
    ratio_accs[r] = metrics["overall_accuracy"]
    print(f"{ratio_accs[r]:.1f}%")

print("\n스윕 완료")

In [ ]:
ratios = list(ratio_accs.keys())
accs   = list(ratio_accs.values())
tokens = [int(r * 196) for r in ratios]  # VideoMME SigLIP 14×14=196

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: ratio vs 정확도
axes[0].plot([r*100 for r in ratios], accs, "o-", color="#2196F3", lw=2.5, ms=10)
axes[0].fill_between([r*100 for r in ratios], accs, min(accs)-1, alpha=0.1, color="#2196F3")
axes[0].axhline(ratio_accs[1.0], color="gray", lw=1.5, ls="--", label="Baseline (r=1.0)")
for r, a in zip(ratios, accs):
    axes[0].annotate(f"{a:.1f}%", (r*100, a), textcoords="offset points",
                     xytext=(0, 10), ha="center", fontsize=10)
axes[0].set_xlabel("Gazing Ratio (%)")
axes[0].set_ylabel("정확도 (%)")
axes[0].set_title(f"VideoMME: Ratio vs 정확도 (N={N_SWEEP})", fontsize=12)
axes[0].set_xticks([r*100 for r in ratios])
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# 오른쪽: 토큰 수 vs 정확도 (파레토)
scatter_c = ["#F44336" if r < 0.5 else "#FF9800" if r < 0.75 else "#4CAF50" for r in ratios]
for r, t, a, c in zip(ratios, tokens, accs, scatter_c):
    axes[1].scatter(t, a, color=c, s=160, zorder=5)
    axes[1].annotate(f"r={r}", (t, a), textcoords="offset points",
                     xytext=(8, 0), fontsize=9)
axes[1].plot(tokens, accs, "--", color="gray", alpha=0.5, lw=1)
axes[1].set_xlabel("처리 토큰 수 (196 = 100%)")
axes[1].set_ylabel("정확도 (%)")
axes[1].set_title("파레토 곡선: 토큰 절감 vs 정확도", fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. 지연 시간 분석

AutoGaze ON/OFF 각각 N번 반복 측정합니다.

In [ ]:
# VideoMME 첫 5개 샘플에 대해 latency 측정
N_LAT   = 5
samples_lat = [ds_vmme[i] for i in range(N_LAT)]
lat_ag, lat_base = [], []

for i, s in enumerate(samples_lat):
    frm = load_video_from_bytes(s[task_cfg.video_bytes_col], num_frames=NUM_FRAMES)
    pmt = task_cfg.build_prompt(s)

    _sync(); t0 = time.perf_counter()
    run_nvila(processor_ag, model, frm, pmt)
    _sync(); lat_ag.append((time.perf_counter()-t0)*1000)

    _sync(); t0 = time.perf_counter()
    run_nvila(processor_base, model, frm, pmt)
    _sync(); lat_base.append((time.perf_counter()-t0)*1000)

    print(f"  [{i+1}/{N_LAT}] AG={lat_ag[-1]:.0f}ms  Base={lat_base[-1]:.0f}ms")

print(f"\nmedian  AG  : {np.median(lat_ag):.1f} ms")
print(f"median Base : {np.median(lat_base):.1f} ms")
print(f"절감률       : {100*(np.median(lat_base)-np.median(lat_ag))/np.median(lat_base):.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 샘플별 latency
x = np.arange(N_LAT)
axes[0].plot(x, lat_ag,   "o-", color="#2196F3", lw=2, ms=8, label=f"AutoGaze ON (r={RATIO})")
axes[0].plot(x, lat_base, "s--", color="#607D8B", lw=2, ms=8, label="AutoGaze OFF")
axes[0].set_xlabel("샘플 인덱스")
axes[0].set_ylabel("지연 시간 (ms)")
axes[0].set_title("샘플별 지연 시간", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# 오른쪽: 박스 플롯
bp = axes[1].boxplot(
    [lat_ag, lat_base],
    labels=[f"AutoGaze ON\n(r={RATIO})", "AutoGaze OFF\n(기준선)"],
    patch_artist=True,
    widths=0.5,
)
bp["boxes"][0].set_facecolor("#90CAF9")
bp["boxes"][1].set_facecolor("#B0BEC5")
med_ag   = np.median(lat_ag)
med_base = np.median(lat_base)
axes[1].text(1, med_ag   + 20, f"{med_ag:.0f}ms",   ha="center", fontsize=11, color="#1565C0", fontweight="bold")
axes[1].text(2, med_base + 20, f"{med_base:.0f}ms", ha="center", fontsize=11, color="#455A64", fontweight="bold")
saving = 100 * (med_base - med_ag) / med_base
axes[1].set_ylabel("지연 시간 (ms)")
axes[1].set_title(f"Latency 분포  (절감 {saving:.1f}%)", fontsize=12)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

---
## 10. 전체 실행 가이드

스모크 테스트 확인 후 아래 스크립트로 전체 평가를 실행합니다.

In [ ]:
guide = """
╔══════════════════════════════════════════════════════════════════════╗
║               전체 벤치마크 실행 방법                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  # 1. 전체 실행 (AutoGaze ON + OFF, 8개 벤치마크)                   ║
║  bash scripts/run_benchmarks.sh                                      ║
║                                                                      ║
║  # 2. 특정 벤치마크만                                               ║
║  bash scripts/run_benchmarks.sh --tasks videomme,mvbench             ║
║                                                                      ║
║  # 3. AutoGaze ON만, ratio=0.5                                      ║
║  bash scripts/run_benchmarks.sh --autogaze-only --ratio 0.5         ║
║                                                                      ║
║  # 4. 스모크 테스트 (100샘플, 빠른 검증)                            ║
║  bash scripts/run_benchmarks.sh --max-samples 100                   ║
║                                                                      ║
║  # 5. HLVid (로컬 다운로드 필요)                                    ║
║  bash scripts/download_hlvid.sh data/HLVid                          ║
║  bash scripts/run_benchmarks.sh --tasks hlvid                        ║
║       --hlvid-video-dir data/HLVid/videos                           ║
║                                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  개별 태스크 직접 실행:                                              ║
║  python -m autogaze.eval.run_benchmark                              ║
║      --task videomme                                                 ║
║      --gazing-ratio 0.75                                            ║
║      --output results/videomme_ag075.json                           ║
║      --resume                                                        ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(guide)

In [ ]:
# ── 기존 결과 파일이 있으면 요약 표 출력 ──────────────────────────────────
import glob
from pathlib import Path
from collections import defaultdict

RESULTS_GLOB = "../results/**/*.json"
files = sorted(glob.glob(RESULTS_GLOB, recursive=True))

if not files:
    print("결과 파일 없음. 스크립트 실행 후 다시 확인하세요.")
else:
    by_task = defaultdict(dict)
    for f in files:
        with open(f) as fp:
            try:
                r = json.load(fp)
            except:
                continue
        if "metrics" not in r or "overall_accuracy" not in r["metrics"]:
            continue
        task = r.get("task", Path(f).stem)
        key  = "ag" if r.get("autogaze") else "baseline"
        by_task[task][key] = r["metrics"]["overall_accuracy"]

    TASK_ORDER = [
        "videomme", "videomme_w_sub", "mvbench",
        "nextqa", "egoschema", "mlvu", "longvideobench", "hlvid"
    ]

    print(f"{'Task':25} {'AutoGaze ON':>12} {'Baseline':>10} {'차이':>8}")
    print("-" * 58)
    for task in TASK_ORDER:
        d = by_task.get(task)
        if not d:
            continue
        ag_a  = d.get("ag")
        bl_a  = d.get("baseline")
        ag_s  = f"{ag_a:.2f}%" if ag_a  is not None else "—"
        bl_s  = f"{bl_a:.2f}%" if bl_a  is not None else "—"
        df_s  = f"{ag_a-bl_a:+.2f}%" if (ag_a and bl_a) else "—"
        print(f"{task:25} {ag_s:>12} {bl_s:>10} {df_s:>8}")